# 6. The live monitor: one observation at a time

**What you will do**

1. Feed prices to a `LiveMonitor` day by day and watch what it reports.
2. Check that the day-by-day results are *identical* to the batch run (so nothing from the future leaked in).
3. Understand the alert rule and how to change it.

**Why a "simulated" live mode?** A real feed (broker, data vendor) would only add plumbing. Replaying history one day at a time
tests the identical logic, reproducibly. Nothing after the current day is ever visible to the monitor.

**Before you start:** notebook 3 (demo run) or the full run. About a minute to run (one vine fit per refit day).

In [1]:
import os, warnings
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":      # notebooks run from the notebooks/ folder; the project root is one level up
    ROOT = ROOT.parent
os.chdir(ROOT)                    # all paths below are relative to the project folder
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 140); pd.set_option("display.max_columns", 30)
print("working folder:", Path.cwd().name)

working folder: Quant_copula


In [2]:
from vine_risk.config import load_config
from vine_risk.pipeline import find_run, load_prices_and_returns
from vine_risk.rolling import load_results
from vine_risk.change_detection import default_lag

cfg = load_config("configs/default.yaml")
prices, returns = load_prices_and_returns(cfg)
RUN = find_run()
results = load_results(RUN / "checkpoint.jsonl")
window = results[0].n_obs
pos = returns.index.get_indexer(pd.DatetimeIndex([r.timestamp for r in results]))
freq = int(np.median(np.diff(pos)))
print(f"run: {RUN} | window {window}, a fit every {freq} day(s)")

run: data/results/w250 | window 250, a fit every 1 day(s)


## 6.1 Set up the monitor

`MonitorConfig` collects the settings: window, refit frequency, how often to run the permutation test (`scan_step`), the alert
thresholds, and whether to compute VaR/ES. `from_config` takes the common ones from `configs/default.yaml`; the rest can be
overridden. `risk_every_fits=0` switches the (slower) risk numbers off for this demonstration.

In [3]:
from vine_risk.monitor import MonitorConfig, LiveMonitor, ConsoleSink, JsonlSink, records_to_frame, alert_state_from_scans

mcfg = MonitorConfig.from_config(cfg, refit_frequency=freq, scan_step=5, n_perm=499, risk_every_fits=0)
mcfg = MonitorConfig(**{**mcfg.__dict__, "vine_kwargs": {**mcfg.vine_kwargs, "num_threads": 4}})   # use 4 CPU threads per fit
print(f"alert starts when p <= {mcfg.alpha} and the distance is >= {mcfg.enter_ratio}x the no-change level; "
      f"ends when it falls below {mcfg.exit_ratio}x")

alert starts when p <= 0.01 and the distance is >= 2.0x the no-change level; ends when it falls below 1.5x


## 6.2 Resume from saved state

A live system does not start from nothing: it loads its saved state. We **prime** the monitor with the returns up to 15 days ago
and with the fits already made. The alert state must be restored as well, otherwise an alert that has been on for months
would be announced as new; we derive it from the stored permutation tests if they exist.

In [4]:
N = 15
history, replay = returns.iloc[:-N], returns.iloc[-N:]
prior = [r for r in results if pd.Timestamp(r.timestamp) <= history.index[-1]]

state = None
scan_file = RUN / "change_scan.parquet"
if scan_file.exists():
    scan = pd.read_parquet(scan_file); scan.index = pd.DatetimeIndex(scan.index)
    state = alert_state_from_scans(scan, mcfg, history.index[-1])

LOG = RUN / "live_notebook" / "log.jsonl"; LOG.parent.mkdir(exist_ok=True); LOG.unlink(missing_ok=True)
monitor = LiveMonitor(list(prices.columns), mcfg, sinks=[ConsoleSink(every=1), JsonlSink(LOG)])
monitor.prime(history, prior, alert_state=state)
monitor.set_last_price(prices.loc[history.index[-1]])      # the last known price, needed to turn the next price into a return
print(f"primed with {len(history)} days of history and {len(prior)} saved fits; alert state: {monitor.alert_state}")

primed with 5456 days of history and 5207 saved fits; alert state: alert


## 6.3 Feed it prices

For every new day we call `on_prices(date, prices_of_that_day)`. Inside, the monitor computes the log return, updates the
window, refits if the schedule says so, computes the metrics, compares with the previous model and the one a window earlier, and
runs the permutation test on scan days. The console sink prints a line for every day here.

In [5]:
for date in replay.index:
    monitor.on_prices(date, prices.loc[date])
live = records_to_frame(monitor.records)
live[["refit", "alert_state", "m_d_t", "s_s_tau", "scan_ratio_tau", "scan_p_tau"]].round(3)

2026-09-14 [alert] D_t=0.132


2026-09-15 [alert] D_t=0.133


2026-09-16 [alert] D_t=0.132


2026-09-17 [alert] D_t=0.134


2026-09-18 [alert] D_t=0.133


...

2026-09-29 [alert] D_t=0.141


2026-09-30 [alert] D_t=0.140


2026-10-01 [alert] D_t=0.140


2026-10-02 [alert] D_t=0.140


,refit,alert_state,m_d_t,s_s_tau,scan_ratio_tau,scan_p_tau
timestamp,,,,,,
2026-09-14,True,alert,0.132,1.140,NaN,NaN
2026-09-15,True,alert,0.133,1.143,NaN,NaN
2026-09-16,True,alert,0.132,1.135,NaN,NaN
2026-09-17,True,alert,0.134,1.136,2.164,0.004
2026-09-18,True,alert,0.133,1.117,NaN,NaN
2026-09-21,True,alert,0.136,1.107,NaN,NaN
2026-09-22,True,alert,0.136,1.119,NaN,NaN
2026-09-23,True,alert,0.136,1.126,NaN,NaN
2026-09-24,True,alert,0.137,1.119,2.059,0.002


Each row is one day. `refit` says whether a vine was fitted that day (`False` on days between refits when the demo run
uses `refit_frequency=5`). Columns starting with `m_` are dependence metrics, `s_` are distances to the model one window earlier,
`scan_` are permutation-test results (only on scan days). The JSON log written by the `JsonlSink` has the same rows:

In [6]:
print(LOG, "-", sum(1 for _ in open(LOG)), "lines")
print(open(LOG).readline()[:300], "...")

data/results/w250/live_notebook/log.jsonl - 15 lines
{"timestamp": "2026-09-14", "position": 5456, "refit": true, "fit_status": "ok", "alert_state": "alert", "new_alert": false, "message": "", "m_d_t": 0.13216982214572584, "m_mean_spearman": 0.07593194862546372, "m_mean_pearson": 0.06552421368242224, "m_mean_model_tau": 0.04948575604301959, "m_mean_lo ...


## 6.4 Same answers as the batch run

The decisive test. The stored batch run (fitted all at once, long before) contains the same dates. If the monitor really sees
only the past, its numbers must agree exactly:

In [7]:
from vine_risk.dependence import dependence_metrics
from vine_risk.change_detection import structural_change_scores

batch_m = dependence_metrics(results)
batch_s = structural_change_scores(results, lag=default_lag(window, freq), baseline=max(20, 500 // freq))
rows = []
for name, mine, theirs in (("D_t", live.m_d_t, batch_m.d_t), ("BIC", live.m_bic, batch_m.bic),
                           ("distance to one window earlier", live.s_s_tau, batch_s.s_tau)):
    mine = mine.dropna(); common = mine.index.intersection(theirs.dropna().index)
    rows.append({"quantity": name, "dates compared": len(common),
                 "largest difference": float(np.abs(mine.loc[common] - theirs.loc[common]).max())})
pd.DataFrame(rows).set_index("quantity")

,dates compared,largest difference
quantity,,
D_t,15,0.0
BIC,15,0.0
distance to one window earlier,15,0.0


Differences of zero (or of rounding size, like 1e-17) mean the monitor and the batch pipeline are the same computation.
`tests/test_monitor.py` also checks that running on a prefix of the data gives the same records as the full run, and that
resuming from a checkpoint gives the same records as running continuously.

## 6.5 The alert rule

The rule is a small function you can try out. Arguments: the current state, the p-value and the effect size (distance relative to
the no-change level):

In [8]:
from vine_risk.monitor import next_alert_state, NORMAL, ALERT

cases = [("normal", 0.001, 2.5), ("normal", 0.001, 1.2), ("normal", 0.20, 3.0), ("alert", 0.30, 1.7), ("alert", 0.001, 1.3)]
out = []
for state, p, ratio in cases:
    new_state, new_alert, msg = next_alert_state(state, p, ratio, mcfg)
    out.append({"state": state, "p-value": p, "effect": ratio, "→ state": new_state, "new alert": new_alert})
pd.DataFrame(out)

,state,p-value,effect,→ state,new alert
0,normal,0.001,2.5,alert,True
1,normal,0.001,1.2,normal,False
2,normal,0.200,3.0,normal,False
3,alert,0.300,1.7,alert,False
4,alert,0.001,1.3,normal,False


Rows 2 and 3 show why both conditions are needed: a significant but small effect (row 2) does not start an alert, and a large
effect that is not significant (row 3) does not either; row 1 is a start. Row 4 shows the hysteresis: once on, the alert stays on until the effect
falls below the exit level (1.5), even if the p-value is no longer small.

## 6.6 Longer replays and the dashboard

For longer stretches use the script, which also verifies against the batch run and saves a log that the dashboard displays:

```bash
python scripts/run_live.py --days 120
python scripts/run_live.py --start 2023-11-20 --end 2024-02-15      # contains an alert start
streamlit run dashboard/app.py                                       # panel 9 shows the replay
```

**Things to try:** change `mcfg.enter_ratio` to 1.5 and watch alerts start earlier; set `risk_every_fits=1` to get VaR and ES
every day (`live.risk_es_vine`); feed the monitor your own returns with `on_return(date, row)` instead of `on_prices`.

The monitor reports that the dependence of the last year differs from the year before. It does not say why, and it is not a
trading signal.